# 02 — Feature Engineering

Builds all pre-match-safe features from the preprocessed dataset.

**Pre-match safety rule:** every feature in `FEATURE_COLS` must be computable from historical data alone — no current-match stats (kills, map scores, live ratings).

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from conf.settings import PROCESSED_DIR, DATASET_FILE
from src.module.model.data import load_raw_data, preprocess
from src.module.model.features import (
    build_features,
    get_feature_matrix,
    build_team_stats_lookup,
    FEATURE_COLS,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

print(f'Pre-match feature cols: {len(FEATURE_COLS)}')

## 1. Load Preprocessed Dataset

In [ ]:
# Fast path: load from cache if it exists
cached = PROCESSED_DIR / DATASET_FILE
if cached.exists():
    df = pd.read_parquet(cached)
    print(f'Loaded from cache: {df.shape}')
else:
    raw = load_raw_data()
    df = preprocess(raw)
    print(f'Preprocessed: {df.shape}')

## 2. Build Features

`build_features()` runs 13 modules in sequence. Each module adds a set of columns and logs timing.

Slow modules (O(N) per team over 61k matches):
- `team_history_features` ~140s — rolling win rates + h2h
- `form_features`         ~120s — multi-window date rolling
- `fatigue_features`      ~100s — game load & rest days

In [ ]:
df = build_features(df)
print(f'After feature engineering: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Pre-match feature cols in FEATURE_COLS: {len(FEATURE_COLS)}')

## 3. Inspect FEATURE_COLS

In [ ]:
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'{i:3d}. {col}')

## 4. Feature Matrix (X, y)

In [ ]:
X, y = get_feature_matrix(df)
print(f'X: {X.shape}  y: {y.shape}')
print(f'Class balance (winner=1): {y.mean():.1%}')
X.describe().T

## 5. Feature Correlation with Winner

In [ ]:
corr = X.corrwith(y).sort_values()
corr.plot.barh(figsize=(8, max(6, len(corr) // 3)), title='Feature correlation with winner')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()

## 6. Distribution of Top Features by Outcome

In [ ]:
top_feats = corr.abs().sort_values(ascending=False).head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.flat, top_feats):
    for label, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
        X[y == label][feat].dropna().plot.kde(ax=ax, color=color, label=f'winner={label}')
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('Feature distributions by match outcome', y=1.01)
plt.tight_layout()

## 7. NaN Rate per Feature

In [ ]:
null_pct = X.isnull().mean().sort_values(ascending=False)
print(f'Features with NaN: {(null_pct > 0).sum()} / {len(null_pct)}')
null_pct[null_pct > 0].to_frame('null_pct').style.format('{:.1%}')

## 8. Team Stats Lookup

`build_team_stats_lookup()` extracts each team's most recent pre-match feature vector.
Used by `predict_match()` to look up both teams' historical stats before a match.

In [ ]:
lookup = build_team_stats_lookup(df)
teams = sorted(lookup.keys())
print(f'Teams in lookup: {len(teams)}')
print(f'Stats per team: {len(next(iter(lookup.values())))}')
print('\nSample — stats for', teams[0])
pd.Series(lookup[teams[0]]).sort_index()